# `restore_structure` vs `add_observables`

The documented flat-state pattern calls `add_observables` (which runs `_channel_currents`) and then `step` (which runs `_channel_currents` again).

For HH / Leak, `update_states` does not read membrane currents, so the first pass is redundant. Here we compare:
1. **Official path:** `add_observables` → `step` (currents computed twice)
2. **Split path:** `restore_structure` → `step` (currents once; no placeholders)

`_step_channels_state` only queries membrane currents when they are already present in `states`, so `restore_structure` alone is enough for HH.

In [ ]:
import time

import jax
import jax.numpy as jnp
from jax import jit

import jaxley as jx
from jaxley.channels.hh import HH
from jaxley.integrate import build_init_and_step_fn
from jaxley.utils.dynamics import build_dynamic_state_utils

jax.config.update("jax_platform_name", "cpu")

# 4-compartment HH cable
comp = jx.Compartment()
branch = jx.Branch(comp, ncomp=4)
cell = jx.Cell(branch, parents=[-1])
cell.insert(HH())

t_max = 50.0
delta_t = 0.025
n_steps = int(t_max / delta_t)  # 2000

cell.branch(0).comp(0).record("v")
cell.branch(0).comp(0).stimulate(jx.step_current(5.0, 20.0, 0.05, delta_t, t_max))

rec_inds = cell.recordings.rec_index.to_numpy()
rec_states = cell.recordings.state.to_numpy()
externals = cell.externals.copy()
external_inds = cell.external_inds.copy()
ext_i = externals["i"]  # (n_stim, n_steps)

params = cell.get_parameters()
cell.to_jax()

init_fn, step_fn = build_init_and_step_fn(cell)
(
    remove_observables,
    add_observables,
    flatten,
    unflatten,
    restore_structure,
) = build_dynamic_state_utils(cell)

all_states0, all_params = init_fn(params)
dynamic0 = flatten(remove_observables(all_states0))

structured = restore_structure(unflatten(dynamic0))
print("n_comps:", len(all_states0["v"]))
print("n_steps:", n_steps)
print("restore has currents?", any(k.startswith("i_") for k in structured))
print("restore keys:", sorted(structured.keys()))

In [ ]:
@jit
def step_add(dynamic_states, all_params, i_now):
    """Official path: structure + currents, then step (currents again)."""
    all_states = add_observables(unflatten(dynamic_states), all_params, delta_t)
    all_states = step_fn(
        all_states, all_params, {"i": i_now}, external_inds, delta_t=delta_t
    )
    recs = jnp.asarray(
        [
            all_states[rec_state][rec_ind]
            for rec_state, rec_ind in zip(rec_states, rec_inds)
        ]
    )
    return flatten(remove_observables(all_states)), recs


@jit
def step_restore(dynamic_states, all_params, i_now):
    """Split path: structure only (no currents), then step."""
    all_states = restore_structure(unflatten(dynamic_states))
    all_states = step_fn(
        all_states, all_params, {"i": i_now}, external_inds, delta_t=delta_t
    )
    recs = jnp.asarray(
        [
            all_states[rec_state][rec_ind]
            for rec_state, rec_ind in zip(rec_states, rec_inds)
        ]
    )
    return flatten(remove_observables(all_states)), recs


def rollout(step_dyn, dynamic_states, all_params, n=None):
    n = n_steps if n is None else n
    recordings = []
    for step in range(n):
        dynamic_states, recs = step_dyn(
            dynamic_states, all_params, ext_i[:, step]
        )
        recordings.append(recs)
    return jnp.stack(recordings, axis=0), dynamic_states

## Equality check

In [ ]:
# Warmup compile
_ = step_add(dynamic0, all_params, ext_i[:, 0])
_ = step_restore(dynamic0, all_params, ext_i[:, 0])

recs_add, dyn_add = rollout(step_add, dynamic0, all_params)
recs_restore, dyn_restore = rollout(step_restore, dynamic0, all_params)

print("recordings max |diff|:", float(jnp.max(jnp.abs(recs_add - recs_restore))))
print("dynamic states max |diff|:", float(jnp.max(jnp.abs(dyn_add - dyn_restore))))
print("allclose recordings:", bool(jnp.allclose(recs_add, recs_restore)))
print("allclose dynamic states:", bool(jnp.allclose(dyn_add, dyn_restore)))
assert jnp.allclose(recs_add, recs_restore)
assert jnp.allclose(dyn_add, dyn_restore)
print("OK: trajectories match.")

## Timing

Under **eager** execution the first current pass is real work (~1.3×). Under **JIT**, XLA can DCE the unused first pass for HH, so the gap often disappears.

In [ ]:
def bench(fn, n_reps=3):
    times = []
    for _ in range(n_reps):
        t0 = time.perf_counter()
        fn().block_until_ready()
        times.append(time.perf_counter() - t0)
    return times


n_eager = 200


def step_add_eager(dynamic_states, all_params, i_now):
    all_states = add_observables(unflatten(dynamic_states), all_params, delta_t)
    all_states = step_fn(
        all_states, all_params, {"i": i_now}, external_inds, delta_t=delta_t
    )
    recs = jnp.asarray(
        [all_states[s][i] for s, i in zip(rec_states, rec_inds)]
    )
    return flatten(remove_observables(all_states)), recs


def step_restore_eager(dynamic_states, all_params, i_now):
    all_states = restore_structure(unflatten(dynamic_states))
    all_states = step_fn(
        all_states, all_params, {"i": i_now}, external_inds, delta_t=delta_t
    )
    recs = jnp.asarray(
        [all_states[s][i] for s, i in zip(rec_states, rec_inds)]
    )
    return flatten(remove_observables(all_states)), recs


times_add_e = bench(lambda: rollout(step_add_eager, dynamic0, all_params, n=n_eager)[0])
times_restore_e = bench(
    lambda: rollout(step_restore_eager, dynamic0, all_params, n=n_eager)[0]
)
mean_add_e = sum(times_add_e) / len(times_add_e)
mean_restore_e = sum(times_restore_e) / len(times_restore_e)
print(f"[eager {n_eager} steps] add_observables:   {mean_add_e*1e3:.1f} ms")
print(f"[eager {n_eager} steps] restore_structure: {mean_restore_e*1e3:.1f} ms")
print(f"[eager] speedup (add / restore): {mean_add_e / mean_restore_e:.2f}x")

_ = rollout(step_add, dynamic0, all_params)[0].block_until_ready()
_ = rollout(step_restore, dynamic0, all_params)[0].block_until_ready()
times_add = bench(lambda: rollout(step_add, dynamic0, all_params)[0])
times_restore = bench(lambda: rollout(step_restore, dynamic0, all_params)[0])
mean_add = sum(times_add) / len(times_add)
mean_restore = sum(times_restore) / len(times_restore)
print(f"[jit-loop {n_steps} steps] add_observables:   {mean_add*1e3:.1f} ms")
print(f"[jit-loop {n_steps} steps] restore_structure: {mean_restore*1e3:.1f} ms")
print(f"[jit-loop] speedup (add / restore): {mean_add / mean_restore:.2f}x")

## Timing with `lax.scan`

In [ ]:
def make_scan_rollout(step_dyn):
    def body(dynamic_states, i_now):
        dynamic_states, recs = step_dyn(dynamic_states, all_params, i_now)
        return dynamic_states, recs

    @jit
    def run(dynamic_states):
        _, recs = jax.lax.scan(body, dynamic_states, ext_i.T)
        return recs

    return run


scan_add = make_scan_rollout(step_add)
scan_restore = make_scan_rollout(step_restore)

_ = scan_add(dynamic0).block_until_ready()
_ = scan_restore(dynamic0).block_until_ready()

r_add = scan_add(dynamic0)
r_restore = scan_restore(dynamic0)
print("scan max |diff|:", float(jnp.max(jnp.abs(r_add - r_restore))))
assert jnp.allclose(r_add, r_restore)

times_add = bench(lambda: scan_add(dynamic0), n_reps=5)
times_restore = bench(lambda: scan_restore(dynamic0), n_reps=5)
mean_add = sum(times_add) / len(times_add)
mean_restore = sum(times_restore) / len(times_restore)

print(f"scan add_observables:   {mean_add*1e3:.2f} ms")
print(f"scan restore_structure: {mean_restore*1e3:.2f} ms")
print(f"speedup (add / restore): {mean_add / mean_restore:.2f}x")